# Notebook 02: Data Cleaning

**Project:** City of Boston – Building & Housing Violations Analysis  
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li

## Overview

This notebook transforms the raw violations CSV into a clean, analysis-ready dataset. Steps include:
1. Standardizing column names and data types
2. Parsing dates and extracting temporal features
3. Mapping violation cities to canonical Boston neighborhood names
4. Dropping duplicates and irrelevant columns
5. Handling missing values
6. Saving the cleaned dataset to `data/processed/violations_clean.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'

for d in [RAW_DIR, PROCESSED_DIR, EXTERNAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_DIR / 'violations.csv', low_memory=False)
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
df.head(2)

Loaded 17,172 rows, 24 columns


,case_no,ap_case_defn_key,status_dttm,status,code,value,description,violation_stno,violation_sthigh,violation_street,...,ward,contact_addr1,contact_addr2,contact_city,contact_state,contact_zip,sam_id,latitude,longitude,location
0,V91983,1013,NaN,Closed,121.2,NaN,Unsafe and Dangerous,302,NaN,Sumner,...,01,302 Sumner St,NaN,East Boston,MA,02128,132380.0,42.367679,-71.03658,"(42.36767899977675, -71.03658000178699)"
1,V897990,1013,2026-03-20 15:27:46,Open,107.4,NaN,Failed to comply w permit term,14,NaN,Elder,...,07,58 GAINSBOROUGH ST,NaN,BOSTON,MA,02115,52584.0,42.320230,-71.06334,"(42.32023000030727, -71.06334000096946)"


## 1. Standardize Column Names

We strip whitespace and convert all column names to lowercase snake_case for consistency.

In [2]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.columns.tolist())

['case_no', 'ap_case_defn_key', 'status_dttm', 'status', 'code', 'value', 'description', 'violation_stno', 'violation_sthigh', 'violation_street', 'violation_suffix', 'violation_city', 'violation_state', 'violation_zip', 'ward', 'contact_addr1', 'contact_addr2', 'contact_city', 'contact_state', 'contact_zip', 'sam_id', 'latitude', 'longitude', 'location']


## 2. Parse Dates & Extract Temporal Features

The `status_dttm` column contains datetime strings. We parse it and extract `year`, `month`, and `day_of_week` for temporal analysis.

In [3]:
df['status_dttm'] = pd.to_datetime(df['status_dttm'], errors='coerce')
df['year'] = df['status_dttm'].dt.year
df['month'] = df['status_dttm'].dt.month
df['day_of_week'] = df['status_dttm'].dt.day_name()

print('Records with valid date:', df['status_dttm'].notna().sum())
print('Records missing date:  ', df['status_dttm'].isna().sum())
print('Year range:', df['year'].min(), '–', df['year'].max())

Records with valid date: 17171
Records missing date:   1
Year range: 2009.0 – 2026.0


## 3. Neighborhood Mapping

The `violation_city` field contains city/neighborhood names that are inconsistent (mixed case, abbreviations). We map them to canonical Boston neighborhood names. Records from outside Boston proper are flagged.

In [4]:
# Show raw city values
print('Top 30 violation_city values:')
print(df['violation_city'].value_counts().head(30))

Top 30 violation_city values:
violation_city
Dorchester        4687
Boston            2787
East Boston       1770
Roxbury           1656
Mattapan           889
South Boston       875
Hyde Park          834
Brighton           762
Allston            611
Roslindale         556
Jamaica Plain      484
Charlestown        414
West Roxbury       383
Mission Hill       358
                    22
BOSTON              12
Charlestown/         5
East Boston/         5
ALLSTON              5
Mission Hill/        5
South End            4
Mattapan/            4
ROXBURY              4
Dorchester/          3
Roslindale/          3
Jamaica Plain/       3
Hyde Park/           3
Roxbury/             2
Brighton/            2
DORCHESTER           2
Name: count, dtype: int64


In [9]:
neighborhood_map = {
    # Boston neighborhoods (direct)
    'Allston':           'Allston',
    'Back Bay':          'Back Bay',
    'Beacon Hill':       'Beacon Hill',
    'Brighton':          'Brighton',
    'Charlestown':       'Charlestown',
    'Chinatown':         'Chinatown',
    'Dorchester':        'Dorchester',
    'Downtown':          'Downtown',
    'East Boston':       'East Boston',
    'Fenway':            'Fenway',
    'Hyde Park':         'Hyde Park',
    'Jamaica Plain':     'Jamaica Plain',
    'Mattapan':          'Mattapan',
    'Mission Hill':      'Mission Hill',
    'North End':         'North End',
    'Roslindale':        'Roslindale',
    'Roxbury':           'Roxbury',
    'South Boston':      'South Boston',
    'South End':         'South End',
    'West End':          'West End',
    'West Roxbury':      'West Roxbury',
    # Aliases / common variants
    'Boston':            'Downtown',
    'E Boston':          'East Boston',
    'JP':                'Jamaica Plain',
    'So Boston':         'South Boston',
    'Roxbury Crossing':  'Roxbury',
    'Mattapan ':         'Mattapan',
    'Dorchester ':       'Dorchester',
    'BOSTON':            'Downtown',
    'DORCHESTER':        'Dorchester',
    'EAST BOSTON':       'East Boston',
    'ROXBURY':           'Roxbury',
    'MATTAPAN':          'Mattapan',
    'JAMAICA PLAIN':     'Jamaica Plain',
    'HYDE PARK':         'Hyde Park',
    'ROSLINDALE':        'Roslindale',
    'WEST ROXBURY':      'West Roxbury',
    'CHARLESTOWN':       'Charlestown',
    'BRIGHTON':          'Brighton',
    'ALLSTON':           'Allston',
    'SOUTH BOSTON':      'South Boston',
    'SOUTH END':         'South End',

    'Charlestown/':      'Charlestown',
    'East Boston/':      'East Boston',
    'Mission Hill/':     'Mission Hill',
    'Mattapan/':         'Mattapan',
    'Roxbury/':          'Roxbury',
    'Brighton/':         'Brighton',
    'Fenway/':           'Fenway',
    'Back Bay/':         'Back Bay',
    'Boston/West End':   'West End',
}

# Apply case-insensitive mapping: normalize input then map
city_normalized = df['violation_city'].str.strip().str.title()
df['neighborhood'] = city_normalized.map(neighborhood_map)

# For unmapped entries, try title-cased pass-through if it's a known neighborhood
known_neighborhoods = set(neighborhood_map.values())
mask_unmapped = df['neighborhood'].isna()
df.loc[mask_unmapped, 'neighborhood'] = city_normalized[mask_unmapped].where(
    city_normalized[mask_unmapped].isin(known_neighborhoods)
)

mapped = df['neighborhood'].notna().sum()
total = len(df)
print(f'Mapped to neighborhood: {mapped:,} / {total:,} ({100*mapped/total:.1f}%)')
print('Neighborhood distribution:')
print(df['neighborhood'].value_counts())

#unmapped areas
unmapped = df[df['neighborhood'].isna()]['violation_city'].value_counts()
print(f'Unmapped records: {len(df[df["neighborhood"].isna()])}')
print(unmapped)

Mapped to neighborhood: 17,123 / 17,172 (99.7%)
Neighborhood distribution:
neighborhood
Dorchester       4689
Downtown         2799
East Boston      1775
Roxbury          1662
Mattapan          893
South Boston      875
Hyde Park         835
Brighton          765
Allston           616
Roslindale        556
Jamaica Plain     484
Charlestown       419
West Roxbury      383
Mission Hill      363
South End           4
West End            2
Chinatown           1
Fenway              1
Back Bay            1
Name: count, dtype: int64
Unmapped records: 49
violation_city
                            22
Dorchester/                  3
Roslindale/                  3
Jamaica Plain/               3
Hyde Park/                   3
East Boston//                2
Theater District             1
Financial District           1
Allston/Boston               1
ROXBURY CROSSIN              1
Chestnut Hill                1
Dorchester Center            1
Financial District/          1
NorthEnd/                    

## 4. Select & Rename Relevant Columns

We keep only the columns needed for analysis and rename them for clarity.

In [11]:
keep_cols = [
    'case_no', 'status', 'status_dttm', 'year', 'month', 'day_of_week',
    'code', 'description', 'neighborhood',
    'violation_street', 'violation_zip', 'ward',
    'latitude', 'longitude',
]
df_clean = df[[c for c in keep_cols if c in df.columns]].copy()
print(f'Kept columns: {df_clean.columns.tolist()}')
print(f'Shape after column selection: {df_clean.shape}')

Kept columns: ['case_no', 'status', 'status_dttm', 'year', 'month', 'day_of_week', 'code', 'description', 'neighborhood', 'violation_street', 'violation_zip', 'ward', 'latitude', 'longitude']
Shape after column selection: (17172, 14)


## 5. Handle Missing Values

- Records missing both `latitude`/`longitude` AND `neighborhood` are not spatially usable — we flag them.
- Records missing `status_dttm` retain their other fields and are included in non-temporal analyses.
- `description` NaN values are filled with `'Unknown'`.

In [12]:
df_clean['description'] = df_clean['description'].fillna('Unknown')
df_clean['code'] = df_clean['code'].astype(str).str.strip()

# Convert lat/lon to numeric
df_clean['latitude'] = pd.to_numeric(df_clean['latitude'], errors='coerce')
df_clean['longitude'] = pd.to_numeric(df_clean['longitude'], errors='coerce')

# Flag records that have valid spatial data
df_clean['has_coords'] = df_clean['latitude'].notna() & df_clean['longitude'].notna()
df_clean['has_neighborhood'] = df_clean['neighborhood'].notna()

print('Records with coordinates:', df_clean['has_coords'].sum())
print('Records with neighborhood:', df_clean['has_neighborhood'].sum())

Records with coordinates: 16945
Records with neighborhood: 17123


## 6. Remove Duplicates

We deduplicate on `case_no`, keeping the most recent record per case.

In [13]:
before = len(df_clean)
df_clean = df_clean.sort_values('status_dttm', ascending=False)
df_clean = df_clean.drop_duplicates(subset='case_no', keep='first').reset_index(drop=True)
after = len(df_clean)
print(f'Removed {before - after:,} duplicate case numbers. Remaining: {after:,}')

Removed 258 duplicate case numbers. Remaining: 16,914


## 7. Final Quality Check & Save

We verify the cleaned dataset and write it to `data/processed/violations_clean.csv`.

In [14]:
print('=== Final Cleaned Dataset ===' )
print(f'Shape: {df_clean.shape}')
print(f'Date range: {df_clean["status_dttm"].min()} → {df_clean["status_dttm"].max()}')
print(f'Unique neighborhoods: {df_clean["neighborhood"].nunique()}')
print(f'Unique violation codes: {df_clean["code"].nunique()}')
print()
print('Missing values per column:')
print(df_clean.isnull().sum())

=== Final Cleaned Dataset ===
Shape: (16914, 16)
Date range: 2009-12-01 13:18:47 → 2026-03-20 15:27:46
Unique neighborhoods: 19
Unique violation codes: 485

Missing values per column:
case_no               0
status                0
status_dttm           1
year                  1
month                 1
day_of_week           1
code                  0
description           0
neighborhood         49
violation_street      0
violation_zip         0
ward                  0
latitude            214
longitude           214
has_coords            0
has_neighborhood      0
dtype: int64


In [15]:
out_path = PROCESSED_DIR / 'violations_clean.csv'
df_clean.to_csv(out_path, index=False)
print(f'Saved cleaned data → {out_path}')
df_clean.head(5)

Saved cleaned data → D:\Projects\CS506Project\data\processed\violations_clean.csv


,case_no,status,status_dttm,year,month,day_of_week,code,description,neighborhood,violation_street,violation_zip,ward,latitude,longitude,has_coords,has_neighborhood
0,V897990,Open,2026-03-20 15:27:46,2026.0,3.0,Friday,107.4,Failed to comply w permit term,Dorchester,Elder,02125,07,42.32023,-71.06334,True,True
1,V897956,Open,2026-03-20 10:37:20,2026.0,3.0,Friday,102.8,Maintenance,Dorchester,Quincy,02121,13,42.31383,-71.07762,True,True
2,V897950,Open,2026-03-20 10:22:23,2026.0,3.0,Friday,102.8,Maintenance,Roxbury,Shawmut,02118,09,42.33555,-71.08033,True,True
3,V897919,Open,2026-03-20 08:08:21,2026.0,3.0,Friday,102.8,Maintenance,Dorchester,Harvard,02124,14,42.29614,-71.08163,True,True
4,V844949,Open,2026-03-19 09:30:33,2026.0,3.0,Thursday,105.1,Failure to Obtain Permit,Dorchester,Romsey,02125,13,42.31609,-71.05627,True,True


## 8. Property Assessment Cleaning & Structural Feature Extraction

The property assessment file is parcel-level, while the modeling dataset is neighborhood-year level. Therefore, we clean and aggregate property records into one row per neighborhood.

Cleaning decisions:

- Keep fields related to housing stock: `YR_BUILT`, `YR_REMODEL`, `TOTAL_VALUE`, `LIVING_AREA`, `LAND_SF`, `LU`, and `LU_DESC`.
- Convert currency/number strings into numeric values.
- Remove impossible build years outside `1700-2025`.
- Identify residential properties using land-use codes and descriptions.
- Map property `CITY` labels to project neighborhoods. Broad labels such as `BOSTON` are assigned using ZIP-code patterns learned from the cleaned violations data.
- Aggregate to neighborhood-level structural features.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Make this feature-extraction cell safe to run by itself.
if 'PROJECT_ROOT' not in globals():
    current_dir = Path.cwd().resolve()
    PROJECT_ROOT = current_dir if (current_dir / 'data').exists() else current_dir.parent

RAW_DIR = globals().get('RAW_DIR', PROJECT_ROOT / 'data' / 'raw')
PROCESSED_DIR = globals().get('PROCESSED_DIR', PROJECT_ROOT / 'data' / 'processed')
EXTERNAL_DIR = globals().get('EXTERNAL_DIR', PROJECT_ROOT / 'data' / 'external')

for d in [RAW_DIR, PROCESSED_DIR, EXTERNAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if 'df_clean' not in globals():
    clean_path = PROCESSED_DIR / 'violations_clean.csv'
    if clean_path.exists():
        df_clean = pd.read_csv(clean_path, low_memory=False)
        print(f'Loaded cleaned violations for ZIP mapping: {clean_path}')
    else:
        raise NameError(
            'df_clean is not defined and data/processed/violations_clean.csv does not exist. '
            'Run the earlier cleaning cells first, then rerun this cell.'
        )

property_path = EXTERNAL_DIR / 'boston_property_assessment_data.csv'
structural_external_path = EXTERNAL_DIR / 'neighborhood_structural_features.csv'
structural_processed_path = PROCESSED_DIR / 'neighborhood_structural_features.csv'
summary_path = PROCESSED_DIR / 'property_assessment_feature_summary.csv'

if property_path.exists():
    usecols = [
        'CITY', 'LU', 'LU_DESC', 'YR_BUILT', 'YR_REMODEL',
        'TOTAL_VALUE', 'BLDG_VALUE', 'LAND_VALUE', 'LIVING_AREA',
        'GROSS_AREA', 'LAND_SF', 'RES_UNITS', 'COM_UNITS',
        'ZIP_CODE', 'geocode_latitude', 'geocode_longitude'
    ]
    prop_raw = pd.read_csv(property_path, usecols=usecols, low_memory=False)
    prop = prop_raw.copy()

    stats_path = PROCESSED_DIR / 'neighborhood_stats.csv'
    if stats_path.exists():
        stats_for_filter = pd.read_csv(stats_path)
        eligible_neighborhoods = set(stats_for_filter.loc[stats_for_filter['violation_count'] >= 50, 'neighborhood'])
    else:
        eligible_neighborhoods = set(df_clean['neighborhood'].dropna().unique())

    zip_source = df_clean[df_clean['neighborhood'].isin(eligible_neighborhoods) & df_clean['violation_zip'].notna()].copy()
    zip_source['zip5'] = zip_source['violation_zip'].astype(str).str.extract(r'(\d{5})')[0]
    zip_counts = zip_source.groupby(['zip5', 'neighborhood']).size().reset_index(name='n')
    zip_mode = (
        zip_counts.sort_values(['zip5', 'n'], ascending=[True, False])
        .drop_duplicates('zip5')
        .set_index('zip5')['neighborhood']
        .to_dict()
    )

    city_map = {
        'ALLSTON': 'Allston', 'BRIGHTON': 'Brighton', 'CHARLESTOWN': 'Charlestown',
        'DORCHESTER': 'Dorchester', 'DORCHESTER CENTER': 'Dorchester',
        'EAST BOSTON': 'East Boston', 'HYDE PARK': 'Hyde Park',
        'JAMAICA PLAIN': 'Jamaica Plain', 'MATTAPAN': 'Mattapan',
        'MISSION HILL': 'Mission Hill', 'ROSLINDALE': 'Roslindale',
        'ROXBURY': 'Roxbury', 'ROXBURY CROSSING': 'Roxbury',
        'SOUTH BOSTON': 'South Boston', 'WEST ROXBURY': 'West Roxbury',
    }

    prop['city_clean'] = prop['CITY'].astype(str).str.strip().str.upper()
    prop['zip5'] = prop['ZIP_CODE'].astype(str).str.extract(r'(\d{4,5})')[0].str.zfill(5)
    prop['neighborhood_from_city'] = prop['city_clean'].map(city_map)
    prop['neighborhood_from_zip'] = prop['zip5'].map(zip_mode)
    prop['neighborhood'] = prop['neighborhood_from_city']
    broad_city_mask = prop['city_clean'].isin(['BOSTON', 'ROXBURY CROSSING'])
    prop.loc[broad_city_mask, 'neighborhood'] = prop.loc[broad_city_mask, 'neighborhood_from_zip']
    prop['neighborhood'] = prop['neighborhood'].fillna(prop['neighborhood_from_zip'])
    prop = prop[prop['neighborhood'].isin(eligible_neighborhoods)].copy()

    def clean_numeric(series):
        return pd.to_numeric(
            series.astype(str).str.replace(r'[$,]', '', regex=True).str.strip(),
            errors='coerce'
        )

    numeric_cols = ['TOTAL_VALUE', 'BLDG_VALUE', 'LAND_VALUE', 'LIVING_AREA', 'GROSS_AREA',
                    'LAND_SF', 'RES_UNITS', 'COM_UNITS', 'YR_BUILT', 'YR_REMODEL']
    for col in numeric_cols:
        prop[col] = clean_numeric(prop[col])

    valid_year = prop['YR_BUILT'].between(1700, 2025)
    prop.loc[~valid_year, 'YR_BUILT'] = np.nan
    prop['building_age_2025'] = 2025 - prop['YR_BUILT']
    prop['was_remodeled'] = prop['YR_REMODEL'].between(1700, 2025)

    lu = prop['LU'].astype(str).str.upper().str.strip()
    lu_desc = prop['LU_DESC'].astype(str).str.upper()
    prop['is_residential'] = lu.isin(['R1', 'R2', 'R3', 'R4', 'A', 'CD', 'CM', 'RC']) | lu_desc.str.contains('RES|DWELLING|CONDO|APT|HOUSING', na=False)
    prop['is_condo'] = lu.isin(['CD', 'CM']) | lu_desc.str.contains('CONDO', na=False)
    prop['is_multifamily'] = lu.isin(['R2', 'R3', 'R4', 'A']) | lu_desc.str.contains('TWO-FAM|THREE-FAM|APT|MULTI|HOUSING', na=False)
    prop['is_old_pre_1940'] = prop['YR_BUILT'] < 1940
    prop['is_very_old_pre_1900'] = prop['YR_BUILT'] < 1900

    residential = prop[prop['is_residential']].copy()
    structural = residential.groupby('neighborhood').agg(
        property_count=('LU', 'size'),
        median_building_age=('building_age_2025', 'median'),
        mean_building_age=('building_age_2025', 'mean'),
        share_pre_1940=('is_old_pre_1940', 'mean'),
        share_pre_1900=('is_very_old_pre_1900', 'mean'),
        share_remodeled=('was_remodeled', 'mean'),
        median_total_value=('TOTAL_VALUE', 'median'),
        median_building_value=('BLDG_VALUE', 'median'),
        median_land_value=('LAND_VALUE', 'median'),
        median_living_area=('LIVING_AREA', 'median'),
        median_gross_area=('GROSS_AREA', 'median'),
        median_land_sf=('LAND_SF', 'median'),
        share_condo=('is_condo', 'mean'),
        share_multifamily=('is_multifamily', 'mean'),
    ).reset_index()

    pop_path = PROCESSED_DIR / 'neighborhood_stats.csv'
    if pop_path.exists():
        pop = pd.read_csv(pop_path)[['neighborhood', 'population_2025']]
    else:
        pop = pd.read_csv(EXTERNAL_DIR / 'neighborhood_population.csv')[['neighborhood', 'population_2025']]
    structural = structural.merge(pop, on='neighborhood', how='left')
    structural['properties_per_1000_residents'] = structural['property_count'] / structural['population_2025'] * 1000

    round_cols = [c for c in structural.columns if c not in ['neighborhood', 'property_count', 'population_2025']]
    structural[round_cols] = structural[round_cols].round(3)
    structural = structural.sort_values('neighborhood').reset_index(drop=True)
    structural.to_csv(structural_processed_path, index=False)
    structural.to_csv(structural_external_path, index=False)

    feature_summary = pd.DataFrame({
        'metric': [
            'raw_property_rows', 'rows_after_neighborhood_filter', 'residential_rows_used',
            'neighborhoods_output', 'missing_year_built_pct_residential', 'missing_total_value_pct_residential'
        ],
        'value': [
            len(prop_raw), len(prop), len(residential), structural['neighborhood'].nunique(),
            round(residential['YR_BUILT'].isna().mean() * 100, 2),
            round(residential['TOTAL_VALUE'].isna().mean() * 100, 2),
        ]
    })
    feature_summary.to_csv(summary_path, index=False)

    print(f'Saved structural features to {structural_processed_path}')
    print(f'Saved GitHub-friendly copy to {structural_external_path}')
    display(structural.head())
else:
    print('Property assessment raw file not found; using existing processed structural feature table if available.')
    if structural_external_path.exists():
        display(pd.read_csv(structural_external_path).head())


## Summary

The cleaning pipeline:
- Standardized column names and data types
- Parsed dates and extracted year/month features
- Mapped `violation_city` values to canonical neighborhood labels
- Removed duplicate case numbers
- Flagged records with missing spatial information

The cleaned dataset is saved to `data/processed/violations_clean.csv` and will be the input for all downstream notebooks.